# Song Cover Pipeline — Quy Trình Đầy Đủ
## Từ bài hát gốc → chuyển sang giọng model → bài cover hoàn chỉnh

**Pipeline gồm 3 giai đoạn chính:**

```
[Bài hát gốc (.mp3/.wav/...)]
         ↓
┌──────────────────────────────────────────────────────┐
│  GIAI ĐOẠN 1 — TÁCH ÂM THANH (MDX-Net 3 stage)     │
│                                                      │
│  Stage 1: UVR-MDX-NET-Voc_FT.onnx                   │
│    Bài hát → Vocals  +  Instrumental                 │
│                  ↓                                   │
│  Stage 2: UVR_MDXNET_KARA_2.onnx                    │
│    Vocals → Backup Vocals  +  Main Vocals            │
│                                   ↓                  │
│  Stage 3: Reverb_HQ_By_FoxJoy.onnx                  │
│    Main Vocals → Main Vocals DeReverb (sạch nhất)    │
└──────────────────────────────────────────────────────┘
     ↓                   ↓                ↓
 [Instrumental]   [Backup Vocals]  [Main DeReverb]
                                         ↓
┌─────────────────────────────────────────────┐
│  GIAI ĐOẠN 2 — RVC VOICE CONVERSION        │
│                                             │
│  Hubert encode  → content vectors (768d)    │
│  RMVPE F0 extract → pitch curve             │
│  FAISS retrieval → blend features           │
│  net_g vocoder  → audio giọng đích          │
└─────────────────────────────────────────────┘
         ↓
  [Converted Vocal]
         ↓
┌──────────────────────────────────────────────────┐
│  GIAI ĐOẠN 3 — AUDIO MIXING & EFFECTS           │
│                                                  │
│  Effects chain: Highpass → Compressor → Reverb   │
│  Mix overlay:                                    │
│    Main Vocal (converted) : base -4 dB           │
│    Backup Vocals          : base -6 dB           │
│    Instrumental           : base -7 dB           │
└──────────────────────────────────────────────────┘
         ↓
  [Final Cover (.wav / .mp3)]
```

**Thứ tự chạy:**  
Lần đầu: chạy tuần tự từ Bước 0 đến Bước 10.  
Lần sau (cùng model): dùng **Bước 11 — Quick Run** để chạy toàn pipeline trong 1 lần.

## Yêu cầu hệ thống

| Yêu cầu | Chi tiết |
|---------|----------|
| Working directory | Phải là thư mục `rvc_standalone` |
| Python env | Đã cài `requirements.txt` (PyTorch, fairseq, soundfile, librosa...) |
| Dependencies bổ sung | `onnxruntime-gpu`, `pedalboard`, `pydub` |
| FFmpeg | Trên PATH — dùng để đọc audio |
| GPU (khuyến nghị) | Tách nhạc + infer trên GPU nhanh hơn ~10x |
| MDX-Net models | 3 file `.onnx` trong `assets/mdxnet_models/` — **Bước 0.3 tự kiểm tra và tải về** |
| RVC model | `assets/weights/<tên>.pth` + `assets/hubert/hubert_base.pt` + `assets/rmvpe/rmvpe.pt` |

---
# BƯỚC 0: Kiểm Tra Môi Trường

Chạy toàn bộ phần này trước. Nếu có lỗi ở đây, fix trước khi tiếp tục.

### 0.1 — Working directory và cấu trúc thư mục

In [ ]:
import os
import sys
import pathlib

CWD = pathlib.Path.cwd().resolve()
print("Working directory:", CWD)

thu_muc_can_co = [
    ("infer",              "Code inference pipeline"),
    ("configs",            "Config JSON"),
    ("assets/hubert",      "Hubert model"),
    ("assets/weights",     "RVC model .pth"),
    ("assets/rmvpe",       "RMVPE model"),
]

thieu = []
for thu_muc, mo_ta in thu_muc_can_co:
    ok = (CWD / thu_muc).is_dir()
    print(f"  {'✓' if ok else '✗ THIẾU'}  {thu_muc:25s} — {mo_ta}")
    if not ok:
        thieu.append(thu_muc)

if thieu:
    print("\n❌ Thiếu thư mục. Mở Jupyter từ thư mục rvc_standalone.")
else:
    print("\n✅ Cấu trúc thư mục OK.")

### 0.2 — GPU

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {p.name}  |  VRAM: {p.total_memory/1024**3:.1f} GB")
    print("✅ Sẽ chạy trên GPU.")
else:
    print("⚠️  Không có GPU — chạy trên CPU (chậm hơn ~10x).")

### 0.3 — MDX-Net model files (kiểm tra và tải tự động nếu thiếu)

3 file ONNX được lưu trong `assets/mdxnet_models/` ngay trong `rvc_standalone/`.  
Nếu chưa có, ô bên dưới sẽ **tự tải về từ GitHub** (TRvlvr/model_repo).

| File | Vai trò |
|------|---------|
| `UVR-MDX-NET-Voc_FT.onnx` | Tách Vocals / Instrumental |
| `UVR_MDXNET_KARA_2.onnx` | Tách Main / Backup Vocals |
| `Reverb_HQ_By_FoxJoy.onnx` | De-reverb Main Vocals |

> Mỗi file ~60-200 MB. Chỉ tải một lần, lần sau dùng lại từ cache.

In [ ]:
import pathlib
import sys
import urllib.request

# Thư mục lưu MDX-Net models ngay trong rvc_standalone/
_MDX_DIR = pathlib.Path.cwd().resolve() / "assets" / "mdxnet_models"
_MDX_DIR.mkdir(parents=True, exist_ok=True)

_MDXNET_BASE = "https://github.com/TRvlvr/model_repo/releases/download/all_public_uvr_models/"

_MODELS = [
    ("UVR-MDX-NET-Voc_FT.onnx",  "Tách Vocals / Instrumental"),
    ("UVR_MDXNET_KARA_2.onnx",    "Tách Main / Backup Vocals"),
    ("Reverb_HQ_By_FoxJoy.onnx",  "De-reverb Main Vocals"),
]

print(f"Thư mục MDX-Net models: {_MDX_DIR}")
print()

def _download_with_progress(url, dest):
    """Tải file với progress bar đơn giản."""
    tmp = dest.with_suffix(dest.suffix + ".part")
    downloaded = [0]

    def _progress(block_num, block_size, total_size):
        downloaded[0] += block_size
        if total_size > 0:
            pct = min(downloaded[0] / total_size * 100, 100)
            mb_done = downloaded[0] / 1024**2
            mb_total = total_size / 1024**2
            bar = "█" * int(pct / 5) + "░" * (20 - int(pct / 5))
            print(f"\r     [{bar}] {pct:5.1f}%  {mb_done:.1f}/{mb_total:.1f} MB", end="", flush=True)
        else:
            print(f"\r     Đã tải {downloaded[0]/1024**2:.1f} MB...", end="", flush=True)

    try:
        req = urllib.request.Request(url, headers={"User-Agent": "rvc-notebook/1.0"})
        with urllib.request.urlopen(req, timeout=120) as resp:
            total = int(resp.headers.get("Content-Length", 0))
            with open(tmp, "wb") as f:
                block = 8192
                while True:
                    data = resp.read(block)
                    if not data:
                        break
                    f.write(data)
                    _progress(0, len(data), total)
        print()  # newline sau progress bar
        tmp.rename(dest)
    except Exception as e:
        if tmp.exists():
            tmp.unlink()
        raise e


all_ok = True
for fname, role in _MODELS:
    dest = _MDX_DIR / fname
    if dest.exists():
        size_mb = dest.stat().st_size / 1024**2
        print(f"  ✓  {fname:<35s} {size_mb:6.1f} MB  — {role}")
    else:
        print(f"  ↓  {fname:<35s} THIẾU — đang tải...")
        url = _MDXNET_BASE + fname
        try:
            _download_with_progress(url, dest)
            size_mb = dest.stat().st_size / 1024**2
            print(f"     ✅ Tải xong: {fname}  ({size_mb:.1f} MB)")
        except Exception as e:
            print(f"\n     ❌ Tải thất bại: {e}")
            print(f"        Thử tải thủ công từ: {url}")
            print(f"        Rồi đặt vào: {dest}")
            all_ok = False

print()
if all_ok:
    print("✅ Tất cả MDX-Net models đã sẵn sàng.")
    print(f"   Đường dẫn: {_MDX_DIR}")
else:
    print("❌ Một số model chưa có. Tải thủ công rồi chạy lại ô này.")

### 0.4 — RVC model files

In [ ]:
import pathlib

CWD = pathlib.Path.cwd().resolve()

weights_can_co = [
    ("assets/hubert/hubert_base.pt", "Bắt buộc"),
    ("assets/rmvpe/rmvpe.pt",        "Cần nếu dùng f0_method='rmvpe'"),
]

print("Weights cần thiết:")
for duong_dan, mo_ta in weights_can_co:
    f = CWD / duong_dan
    if f.exists():
        print(f"  ✓  {duong_dan}  ({f.stat().st_size/1024**2:.0f} MB)")
    else:
        print(f"  ✗  {duong_dan}  — THIẾU  ({mo_ta})")

print()
weights_dir = CWD / "assets" / "weights"
pth_files = sorted(weights_dir.glob("*.pth")) if weights_dir.exists() else []
print(f"Model .pth có sẵn ({len(pth_files)} file):")
for f in pth_files:
    print(f"  • {f.name}  ({f.stat().st_size/1024**2:.0f} MB)")
if not pth_files:
    print("  (trống — chưa có model)")

print()
indices_dir = CWD / "assets" / "indices"
idx_files = list(indices_dir.glob("*.index")) if indices_dir.exists() else []
idx_files += list((CWD / "logs").rglob("added_*.index")) if (CWD / "logs").exists() else []
print(f"FAISS index có sẵn ({len(idx_files)} file):")
for f in sorted(set(idx_files)):
    rel = f.relative_to(CWD) if f.is_relative_to(CWD) else f
    print(f"  • {rel}  ({f.stat().st_size/1024**2:.1f} MB)")
if not idx_files:
    print("  (trống — infer không dùng index)")

### 0.5 — FFmpeg

In [ ]:
import subprocess

try:
    r = subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True, timeout=5)
    print("✅ FFmpeg OK:", r.stdout.splitlines()[0])
except FileNotFoundError:
    print("❌ FFmpeg không tìm thấy trên PATH.")

### 0.6 — Dependencies bổ sung (onnxruntime, pedalboard, pydub)

In [ ]:
dep_list = [
    ("onnxruntime",  "Chạy MDX-Net ONNX models"),
    ("pedalboard",   "Audio effects chain (Highpass, Compressor, Reverb)"),
    ("pydub",        "Mix overlay các tracks"),
    ("librosa",      "Load audio"),
    ("soundfile",    "Ghi file WAV"),
    ("tqdm",         "Progress bar"),
]

all_ok = True
for pkg, mo_ta in dep_list:
    try:
        mod = __import__(pkg)
        ver = getattr(mod, "__version__", "?")
        print(f"  ✓  {pkg:<15s} {ver}  — {mo_ta}")
    except ImportError:
        print(f"  ✗  {pkg:<15s} THIẾU  — {mo_ta}")
        print(f"       → pip install {pkg}")
        all_ok = False

print()
if all_ok:
    print("✅ Tất cả dependencies OK.")
else:
    print("❌ Cài các package thiếu rồi restart kernel.")

---
# BƯỚC 1: Cấu Hình Đầu Vào

**Chỉ cần sửa phần này.** Tất cả các bước sau đều dùng các biến được đặt ở đây.

> Sau khi sửa xong, chạy ô này rồi chạy tuần tự từ Bước 2 xuống.

In [ ]:
# ================================================================
# ĐẦU VÀO
# ================================================================

# File bài hát gốc (hỗ trợ .wav .mp3 .flac .m4a .ogg)
INPUT_SONG = r"C:\đường\dẫn\đến\bai_hat_goc.mp3"  # <-- SỬA

# Thư mục lưu tất cả kết quả trung gian và file cuối
OUTPUT_DIR = "pipeline_output"  # Sẽ được tạo tự động nếu chưa có

# Thư mục chứa các file ONNX của MDX-Net (trong rvc_standalone/assets/mdxnet_models/)
# Bước 0.3 sẽ tự tải về đây nếu thiếu — không cần sửa
MDX_MODELS_DIR = "assets/mdxnet_models"

# ================================================================
# RVC MODEL
# ================================================================

# Tên file model .pth trong assets/weights/
MODEL_FILE = "giong_A_infer.pth"  # <-- SỬA

# Đường dẫn file .index (để rỗng nếu không dùng)
INDEX_PATH = ""  # <-- SỬA: ví dụ r".\assets\indices\giong_A_added_IVF326_Flat_nprobe_1_giong_A_v2.index"

# ================================================================
# THAM SỐ VOICE CONVERSION (xem giải thích chi tiết ở Bước 5)
# ================================================================

SPEAKER_ID    = 0        # Chỉ số speaker (0 nếu model 1 giọng)
F0_UP_KEY     = 0        # Pitch shift (semitone). Nam→Nữ: +5~+7, Nữ→Nam: -5~-7
F0_METHOD     = "rmvpe" # "rmvpe" (tốt nhất) | "harvest" | "crepe" | "pm"
INDEX_RATE    = 0.75     # 0.0-1.0: tỉ lệ blend FAISS index (0 = không dùng)
FILTER_RADIUS = 3        # 0-7: lọc F0 (chỉ tác dụng với harvest)
RESAMPLE_SR   = 0        # 0 = giữ sr model | 44100 | 48000
RMS_MIX_RATE  = 0.25     # 0.0-1.0: blend envelope âm lượng từ nguồn
PROTECT       = 0.33     # 0.0-0.5: bảo vệ phụ âm (0=tối đa, 0.5=tắt)

# ================================================================
# THAM SỐ TÁCH ÂM THANH
# ================================================================

DENOISE = True  # True: denoise khi chạy MDX-Net (khuyến nghị, chất lượng cao hơn)

# ================================================================
# THAM SỐ AUDIO MIXING (xem giải thích chi tiết ở Bước 8)
# ================================================================

# Effects chain áp dụng lên vocal converted
REVERB_ROOM_SIZE = 0.15  # 0.0-1.0: kích thước phòng (càng cao càng rộng)
REVERB_WET       = 0.2   # 0.0-1.0: mức độ reverb (wet signal)
REVERB_DRY       = 0.8   # 0.0-1.0: mức độ dry signal
REVERB_DAMPING   = 0.7   # 0.0-1.0: damping tần số cao trong reverb

# Điều chỉnh âm lượng (dB) thêm vào gain mặc định
# Gain mặc định: Main=-4dB, Backup=-6dB, Instrumental=-7dB
MAIN_GAIN   = 0   # +/- dB cho main vocal (converted)
BACKUP_GAIN = 0   # +/- dB cho backup vocal
INST_GAIN   = 0   # +/- dB cho instrumental

OUTPUT_FORMAT = "wav"  # "wav" hoặc "mp3"

# ================================================================

print("Cấu hình đầu vào:")
print(f"  INPUT_SONG    : {INPUT_SONG}")
print(f"  OUTPUT_DIR    : {OUTPUT_DIR}")
print(f"  MDX_MODELS_DIR: {MDX_MODELS_DIR}  (assets/mdxnet_models trong rvc_standalone/)")
print(f"  MODEL_FILE    : {MODEL_FILE}")
print(f"  INDEX_PATH    : {INDEX_PATH if INDEX_PATH else '(không dùng)'}")
print()
print(f"  F0_UP_KEY  = {F0_UP_KEY}  |  F0_METHOD = {F0_METHOD}")
print(f"  INDEX_RATE = {INDEX_RATE} |  RMS_MIX   = {RMS_MIX_RATE}  |  PROTECT = {PROTECT}")
print()
print(f"  DENOISE         = {DENOISE}")
print(f"  REVERB_ROOM     = {REVERB_ROOM_SIZE}  WET={REVERB_WET}  DRY={REVERB_DRY}  DAMP={REVERB_DAMPING}")
print(f"  GAINS: Main={MAIN_GAIN:+}dB  Backup={BACKUP_GAIN:+}dB  Inst={INST_GAIN:+}dB")
print(f"  OUTPUT_FORMAT   = {OUTPUT_FORMAT}")

---
# BƯỚC 2: Khởi Tạo Môi Trường Python

Chạy **một lần** sau khi mở notebook hoặc restart kernel.

In [ ]:
import os
import sys
import pathlib

STANDALONE_ROOT = pathlib.Path.cwd().resolve()
if not (STANDALONE_ROOT / "infer" / "modules" / "vc" / "modules.py").is_file():
    raise SystemExit(
        "\n❌ Working directory sai!"
        "\n   Phải chạy từ thư mục rvc_standalone."
    )

os.chdir(STANDALONE_ROOT)
if str(STANDALONE_ROOT) not in sys.path:
    sys.path.insert(0, str(STANDALONE_ROOT))

os.environ.setdefault("weight_root",       "assets/weights")
os.environ.setdefault("index_root",         "logs")
os.environ.setdefault("outside_index_root", "assets/indices")
os.environ.setdefault("rmvpe_root",         "assets/rmvpe")

try:
    from dotenv import load_dotenv
    load_dotenv(os.path.join(STANDALONE_ROOT, ".env"), override=False)
except ImportError:
    pass

# Tạo thư mục output
output_dir = STANDALONE_ROOT / OUTPUT_DIR
output_dir.mkdir(parents=True, exist_ok=True)

# Kiểm tra file input
import pathlib as _pl
input_song_path = _pl.Path(INPUT_SONG)
if not input_song_path.exists():
    print(f"❌ File bài hát không tồn tại: {INPUT_SONG}")
    print("   Sửa INPUT_SONG ở Bước 1.")
else:
    song_stem = input_song_path.stem
    print(f"✅ Input: {input_song_path.name}  ({input_song_path.stat().st_size/1024**2:.1f} MB)")
    print(f"   song_stem = '{song_stem}' (dùng làm prefix cho các file output)")

print(f"\n✅ Output directory: {output_dir}")
print(f"✅ Python path OK. Root: {STANDALONE_ROOT}")

---
# BƯỚC 3: Tách Âm Thanh — MDX-Net 3 Stage

**Tại sao cần tách âm thanh?**  
RVC chuyển đổi giọng hiệu quả nhất khi input là **giọng sạch, không có nhạc nền**. Nếu infer trực tiếp từ bài hát gốc, nhạc nền sẽ bị xử lý cùng và gây artifact.

**Pipeline 3 stage:**

| Stage | Model | Input → Output |
|-------|-------|----------------|
| 1 | `UVR-MDX-NET-Voc_FT.onnx` | Bài hát → **Vocals** + **Instrumental** |
| 2 | `UVR_MDXNET_KARA_2.onnx` | Vocals → **Backup Vocals** + **Main Vocals** |
| 3 | `Reverb_HQ_By_FoxJoy.onnx` | Main Vocals → **Main Vocals DeReverb** |

**`DENOISE=True`** chạy model 2 lần (`process(wave)` - `process(-wave)`) rồi lấy trung bình → giảm nhiễu đáng kể, nhưng chậm gấp đôi.

### 3.1 — Định nghĩa MDX-Net Classes (chỉ cần chạy 1 lần)

In [ ]:
import gc
import hashlib
import os
import queue
import threading

import librosa
import numpy as np
import onnxruntime as ort
import soundfile as sf
import torch
from tqdm import tqdm

# Bảng tham số cho từng model ONNX (xác định qua MD5 hash của file)
_MDX_MODEL_PARAMS = {
    "77d07b2667ddf05b9e3175941b4454a0": {   # UVR-MDX-NET-Voc_FT.onnx
        "compensate": 1.021, "mdx_dim_f_set": 3072,
        "mdx_dim_t_set": 8,  "mdx_n_fft_scale_set": 7680, "primary_stem": "Vocals",
    },
    "1d64a6d2c30f709b8c9b4ce1366d96ee": {   # UVR_MDXNET_KARA_2.onnx
        "compensate": 1.035, "mdx_dim_f_set": 2048,
        "mdx_dim_t_set": 8,  "mdx_n_fft_scale_set": 5120, "primary_stem": "Instrumental",
    },
    "cd5b2989ad863f116c855db1dfe24e39": {   # Reverb_HQ_By_FoxJoy.onnx
        "compensate": 1.035, "mdx_dim_f_set": 3072,
        "mdx_dim_t_set": 9,  "mdx_n_fft_scale_set": 6144, "primary_stem": "Other",
    },
}

_STEM_NAMING = {
    "Vocals": "Instrumental", "Instrumental": "Vocals",
    "Other": "Instruments",   "Drums": "Drumless", "Bass": "Bassless",
}


class _MDXModel:
    """Thông số mô hình + STFT/iSTFT."""
    def __init__(self, device, dim_f, dim_t, n_fft, hop=1024, stem_name=None, compensation=1.0):
        self.dim_f = dim_f
        self.dim_t = dim_t
        self.dim_c = 4
        self.n_fft = n_fft
        self.hop = hop
        self.stem_name = stem_name
        self.compensation = compensation
        self.n_bins = n_fft // 2 + 1
        self.chunk_size = hop * (dim_t - 1)
        self.window = torch.hann_window(window_length=n_fft, periodic=True).to(device)
        self.freq_pad = torch.zeros([1, 4, self.n_bins - dim_f, dim_t]).to(device)

    def stft(self, x):
        x = x.reshape([-1, self.chunk_size])
        x = torch.stft(x, n_fft=self.n_fft, hop_length=self.hop,
                        window=self.window, center=True, return_complex=True)
        x = torch.view_as_real(x)
        x = x.permute([0, 3, 1, 2])
        x = x.reshape([-1, 2, 2, self.n_bins, self.dim_t]).reshape([-1, 4, self.n_bins, self.dim_t])
        return x[:, :, : self.dim_f]

    def istft(self, x, freq_pad=None):
        freq_pad = self.freq_pad.repeat([x.shape[0], 1, 1, 1]) if freq_pad is None else freq_pad
        x = torch.cat([x, freq_pad], -2)
        x = x.reshape([-1, 2, 2, self.n_bins, self.dim_t]).reshape([-1, 2, self.n_bins, self.dim_t])
        x = x.permute([0, 2, 3, 1]).contiguous()
        x = torch.view_as_complex(x)
        x = torch.istft(x, n_fft=self.n_fft, hop_length=self.hop,
                         window=self.window, center=True)
        return x.reshape([-1, 2, self.chunk_size])


class _MDX:
    """MDX-Net inference session."""
    def __init__(self, model_path, params, providers):
        self.device = params.window.device
        self.model = params
        self.ort = ort.InferenceSession(model_path, providers=providers)
        # Warmup
        self.ort.run(None, {"input": torch.rand(1, 4, params.dim_f, params.dim_t).numpy()})
        self.process = lambda spec: self.ort.run(None, {"input": spec.cpu().numpy()})[0]
        self.prog = None

    @staticmethod
    def get_hash(model_path):
        try:
            with open(model_path, "rb") as f:
                f.seek(-10000 * 1024, 2)
                return hashlib.md5(f.read()).hexdigest()
        except Exception:
            with open(model_path, "rb") as f:
                return hashlib.md5(f.read()).hexdigest()

    @staticmethod
    def _segment(wave, combine=True, chunk_size=0, margin_size=44100):
        if combine:
            result = None
            for i, seg in enumerate(wave):
                start = 0 if i == 0 else margin_size
                end = None if i == len(wave) - 1 else -margin_size
                if margin_size == 0: end = None
                result = seg[:, start:end] if result is None else np.concatenate((result, seg[:, start:end]), axis=-1)
            return result
        else:
            n = wave.shape[-1]
            if chunk_size <= 0 or chunk_size > n: chunk_size = n
            if margin_size > chunk_size: margin_size = chunk_size
            segs = []
            for i, skip in enumerate(range(0, n, chunk_size)):
                margin = 0 if i == 0 else margin_size
                end = min(skip + chunk_size + margin_size, n)
                segs.append(wave[:, skip - margin:end].copy())
                if end == n: break
            return segs

    def _pad_wave(self, wave):
        n = wave.shape[1]
        trim = self.model.n_fft // 2
        gen_size = self.model.chunk_size - 2 * trim
        pad = gen_size - n % gen_size
        wave_p = np.concatenate((np.zeros((2, trim)), wave, np.zeros((2, pad)), np.zeros((2, trim))), 1)
        mix_waves = []
        for i in range(0, n + pad, gen_size):
            mix_waves.append(np.array(wave_p[:, i: i + self.model.chunk_size]))
        return torch.tensor(mix_waves, dtype=torch.float32).to(self.device), pad, trim

    def _process_wave(self, mix_waves, trim, pad, q, err_q, _id):
        try:
            pw = []
            with torch.no_grad():
                for w in mix_waves.split(1):
                    self.prog.update()
                    spec = self.model.stft(w)
                    proc = torch.tensor(self.process(spec))
                    out = self.model.istft(proc.to(self.device))
                    pw.append(out[:, :, trim:-trim].transpose(0, 1).reshape(2, -1).cpu().numpy())
            sig = np.concatenate(pw, axis=-1)
            if pad: sig = sig[:, :-pad]
            q.put({_id: sig})
        except Exception as exc:
            err_q.put(exc)

    def process_wave(self, wave, mt_threads=1):
        self.prog = tqdm(total=0, disable=True)
        chunk = wave.shape[-1] // mt_threads
        waves = self._segment(wave, False, chunk)
        q, err_q = queue.Queue(), queue.Queue()
        threads = []
        for c, batch in enumerate(waves):
            mix_waves, pad, trim = self._pad_wave(batch)
            self.prog.total = len(mix_waves) * mt_threads
            t = threading.Thread(target=self._process_wave, args=(mix_waves, trim, pad, q, err_q, c))
            t.start()
            threads.append(t)
        for t in threads: t.join()
        self.prog.close()
        if not err_q.empty(): raise err_q.get()
        batches = [list(v.values())[0] for v in sorted(q.queue, key=lambda d: list(d.keys())[0])]
        return self._segment(batches, True, chunk)


def _onnx_providers():
    if torch.cuda.is_available():
        avail = set(ort.get_available_providers())
        if "CUDAExecutionProvider" in avail:
            return ["CUDAExecutionProvider", "CPUExecutionProvider"]
    return ["CPUExecutionProvider"]


def _to_stereo(audio_path):
    wave, sr = librosa.load(audio_path, mono=False, sr=44100)
    if wave.ndim == 1:
        stereo_path = str(audio_path).replace(".wav", "_stereo.wav")
        sf.write(stereo_path, np.stack([wave, wave], axis=0).T, sr)
        return stereo_path
    return audio_path


def run_mdx(
    output_dir, model_path, input_file,
    exclude_main=False, exclude_inversion=False,
    suffix=None, invert_suffix=None,
    denoise=False,
):
    """
    Chạy một model MDX-Net ONNX.

    Trả về: (main_path, invert_path)
      main_path   = file stem chính của model (ví dụ Vocals.wav)
      invert_path = file inversion (= input - main)  (ví dụ Instrumental.wav)
    """
    device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
    m_threads = 1
    if torch.cuda.is_available():
        vram = torch.cuda.get_device_properties(device).total_memory / 1024 ** 3
        m_threads = 2 if vram >= 8 else 1

    model_hash = _MDX.get_hash(str(model_path))
    mp = _MDX_MODEL_PARAMS.get(model_hash)
    if mp is None:
        raise ValueError(f"Không tìm thấy tham số MDX cho model hash {model_hash}.")

    params = _MDXModel(
        device,
        dim_f=mp["mdx_dim_f_set"],
        dim_t=2 ** mp["mdx_dim_t_set"],
        n_fft=mp["mdx_n_fft_scale_set"],
        stem_name=mp["primary_stem"],
        compensation=mp["compensate"],
    )
    sess = _MDX(str(model_path), params, providers=_onnx_providers())

    stereo_input = _to_stereo(str(input_file))
    wave, sr = librosa.load(stereo_input, mono=False, sr=44100)
    if wave.ndim == 1:
        wave = np.stack([wave, wave], axis=0)

    peak = max(np.max(wave), abs(np.min(wave)))
    if peak > 0: wave /= peak

    if denoise:
        processed = -(sess.process_wave(-wave, m_threads)) + sess.process_wave(wave, m_threads)
        processed *= 0.5
    else:
        processed = sess.process_wave(wave, m_threads)
    processed *= peak

    stem_name = params.stem_name if suffix is None else suffix
    os.makedirs(output_dir, exist_ok=True)
    in_stem = os.path.splitext(os.path.basename(str(input_file)))[0]

    main_path = None
    if not exclude_main:
        main_path = os.path.join(output_dir, f"{in_stem}_{stem_name}.wav")
        sf.write(main_path, processed.T, sr)

    invert_path = None
    if not exclude_inversion:
        inv_name = (_STEM_NAMING.get(stem_name) if invert_suffix is None else invert_suffix)
        if inv_name is None: inv_name = f"{stem_name}_diff"
        invert_path = os.path.join(output_dir, f"{in_stem}_{inv_name}.wav")
        sf.write(invert_path, (-processed.T * params.compensation) + wave.T, sr)

    del sess, processed, wave
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    return main_path, invert_path


print("✅ MDX-Net classes đã sẵn sàng.")
print(f"   ONNX providers: {_onnx_providers()}")

### 3.2 — Stage 1: Tách Instrumental + Vocals

Model `UVR-MDX-NET-Voc_FT.onnx` phân tách bài hát thành:
- **Vocals**: toàn bộ giọng hát (main + backup)
- **Instrumental**: nhạc nền (nhạc cụ, không có giọng)

In [ ]:
import pathlib
import time

mdx_dir = pathlib.Path(MDX_MODELS_DIR).resolve()
sep_out = str(output_dir / "separation")

model_vocal = mdx_dir / "UVR-MDX-NET-Voc_FT.onnx"
if not model_vocal.exists():
    raise FileNotFoundError(f"Không tìm thấy model: {model_vocal}")

print(f"Stage 1: {model_vocal.name}")
print(f"Input  : {INPUT_SONG}")
print(f"Denoise: {DENOISE}")
print("Đang chạy...")

t0 = time.time()
vocals_path, instrumental_path = run_mdx(
    output_dir=sep_out,
    model_path=model_vocal,
    input_file=INPUT_SONG,
    denoise=DENOISE,
)
elapsed = time.time() - t0

print(f"\n✅ Stage 1 xong ({elapsed:.1f}s)")
print(f"   Vocals        → {vocals_path}")
print(f"   Instrumental  → {instrumental_path}")

### 3.3 — Stage 2: Tách Backup Vocals + Main Vocals

Model `UVR_MDXNET_KARA_2.onnx` tách tiếp từ Vocals:
- **Backup Vocals**: các giọng phụ, hòa âm, background
- **Main Vocals**: giọng chính của ca sĩ (sạch nhất, dùng để infer)

In [ ]:
import time

model_karaoke = mdx_dir / "UVR_MDXNET_KARA_2.onnx"
if not model_karaoke.exists():
    raise FileNotFoundError(f"Không tìm thấy model: {model_karaoke}")

print(f"Stage 2: {model_karaoke.name}")
print(f"Input  : {vocals_path}")
print("Đang chạy...")

t0 = time.time()
backup_vocal_path, main_vocal_path = run_mdx(
    output_dir=sep_out,
    model_path=model_karaoke,
    input_file=vocals_path,
    suffix="Backup",
    invert_suffix="Main",
    denoise=DENOISE,
)
elapsed = time.time() - t0

# Dọn file tạm (Vocals tổng hợp không cần nữa)
import os
if vocals_path and os.path.exists(vocals_path):
    os.remove(vocals_path)
    print(f"   [dọn] Xóa file tạm: {os.path.basename(vocals_path)}")

print(f"\n✅ Stage 2 xong ({elapsed:.1f}s)")
print(f"   Backup Vocals → {backup_vocal_path}")
print(f"   Main Vocals   → {main_vocal_path}")

### 3.4 — Stage 3: De-reverb Main Vocals

Model `Reverb_HQ_By_FoxJoy.onnx` loại bỏ reverb/echo còn sót lại trong Main Vocals:
- **Main Vocals DeReverb**: giọng khô nhất, ít vang nhất → **đây là input cho RVC**

> `exclude_main=True`: không lưu phần reverb ("Other"), chỉ giữ lại phần "DeReverb".

In [ ]:
import time

model_dereverb = mdx_dir / "Reverb_HQ_By_FoxJoy.onnx"
if not model_dereverb.exists():
    raise FileNotFoundError(f"Không tìm thấy model: {model_dereverb}")

print(f"Stage 3: {model_dereverb.name}")
print(f"Input  : {main_vocal_path}")
print("Đang chạy...")

t0 = time.time()
_, main_vocal_dereverb_path = run_mdx(
    output_dir=sep_out,
    model_path=model_dereverb,
    input_file=main_vocal_path,
    invert_suffix="DeReverb",
    exclude_main=True,   # Không lưu phần reverb, chỉ lưu DeReverb
    denoise=DENOISE,
)
elapsed = time.time() - t0

import os
if main_vocal_path and os.path.exists(main_vocal_path):
    os.remove(main_vocal_path)
    print(f"   [dọn] Xóa file tạm: {os.path.basename(main_vocal_path)}")

print(f"\n✅ Stage 3 xong ({elapsed:.1f}s)")
print(f"   Main Vocals DeReverb → {main_vocal_dereverb_path}")
print()
print("=" * 50)
print("TÓM TẮT KẾT QUẢ TÁCH ÂM:")
print("=" * 50)
print(f"  Instrumental       : {os.path.basename(instrumental_path)}")
print(f"  Backup Vocals      : {os.path.basename(backup_vocal_path)}")
print(f"  Main DeReverb (→RVC): {os.path.basename(main_vocal_dereverb_path)}")

### 3.5 — Nghe thử kết quả tách âm (tùy chọn)

In [ ]:
from IPython.display import Audio, display
import pathlib

print("🎵 Bài hát gốc:")
display(Audio(INPUT_SONG))

print("\n🎸 Instrumental (nhạc nền):")
display(Audio(instrumental_path))

print("\n🎤 Main Vocals DeReverb (input cho RVC):")
display(Audio(main_vocal_dereverb_path))

print("\n🎙️ Backup Vocals:")
display(Audio(backup_vocal_path))

---
# BƯỚC 4: Khởi Tạo RVC Inference

Nạp model RVC vào bộ nhớ. Chỉ cần chạy 1 lần sau khi mở notebook hoặc restart kernel.

### 4.1 — Config (phát hiện GPU, FP16, chunk size)

In [ ]:
import sys

_saved_argv = sys.argv
sys.argv = ["rvc_infer"]

try:
    from configs.config import Config
    config = Config()
finally:
    sys.argv = _saved_argv

print("=" * 45)
print("CONFIG")
print("=" * 45)
print(f"  device   : {config.device}")
print(f"  is_half  : {config.is_half}  (FP16)")
print(f"  x_pad    : {config.x_pad}  giây")
print(f"  x_query  : {config.x_query} giây")
print(f"  x_center : {config.x_center} giây")
print(f"  x_max    : {config.x_max}  giây")
print("=" * 45)
print("✅ Config OK.")

### 4.2 — Load model .pth

In [ ]:
from infer.modules.vc.modules import VC

vc = VC(config)
_ = vc.get_vc(MODEL_FILE)

if vc.net_g is None:
    print("❌ Load model thất bại. Kiểm tra tên file và thư mục assets/weights/.")
else:
    print("=" * 45)
    print("THÔNG TIN MODEL")
    print("=" * 45)
    print(f"  File         : {MODEL_FILE}")
    print(f"  Sample rate  : {vc.tgt_sr} Hz")
    print(f"  Version      : {vc.version}")
    print(f"  Có F0 (pitch): {bool(vc.if_f0)}")
    try:
        n_spk = vc.cpt["config"][-3]
        print(f"  Speaker count: {n_spk}  (SPEAKER_ID: 0 đến {n_spk-1})")
    except Exception:
        pass
    print("=" * 45)
    print("✅ Model đã tải.")

### 4.3 — Load Hubert model

Hubert encode audio nguồn thành **content vector** (biểu diễn nội dung âm thanh, không phụ thuộc timbre). Chỉ cần load 1 lần, tái dùng cho nhiều lần infer.

In [ ]:
from infer.modules.vc.utils import load_hubert

print("Đang tải Hubert model...")
vc.hubert_model = load_hubert(config)

device_h = next(vc.hubert_model.parameters()).device
print(f"✅ Hubert đã tải — device: {device_h}")

### 4.4 — Cấu hình FAISS Index (tùy chọn)

FAISS index blend feature từ training data vào kết quả → giọng infer tự nhiên và giống model hơn.  
Để `INDEX_PATH = ""` ở Bước 1 nếu không muốn dùng.

In [ ]:
import pathlib

if not INDEX_PATH:
    print("⚠️  Không dùng FAISS index (INDEX_PATH rỗng).")
    print("   Infer vẫn chạy được. Đặt INDEX_RATE=0 hoặc để INDEX_PATH rỗng.")
else:
    idx_path = pathlib.Path(INDEX_PATH)
    if idx_path.exists():
        print(f"✅ Index: {idx_path.name}  ({idx_path.stat().st_size/1024**2:.1f} MB)")
        print(f"   INDEX_RATE = {INDEX_RATE}  (0=không dùng, 0.75=mặc định, 1.0=hoàn toàn)")
    else:
        print(f"❌ Không tìm thấy: {INDEX_PATH}")
        print("   Kiểm tra lại đường dẫn hoặc để INDEX_PATH rỗng.")

---
# BƯỚC 5: Giải Thích Tham Số Voice Conversion

Tất cả tham số đã được đặt ở **Bước 1**. Phần này chỉ giải thích để bạn hiểu và điều chỉnh nếu cần.

---

### SPEAKER_ID — Chỉ số Speaker
- Luôn để `0` nếu model được train 1 giọng (99% trường hợp).
- Nếu model multi-speaker: chọn ID từ 0 đến `n_spk - 1` (xem Bước 4.2).

---

### F0_UP_KEY — Dịch Tông (Pitch Shift)
- Đơn vị: **semitone** (nửa cung)
- `0` = giữ nguyên tông | `+12` = tăng 1 quãng tám | `-12` = giảm 1 quãng tám

| Trường hợp | Giá trị gợi ý |
|------------|---------------|
| Giọng cùng giới tính | `0` hoặc `±1-3` |
| Nam → Nữ | `+5` đến `+7` |
| Nữ → Nam | `-5` đến `-7` |

---

### F0_METHOD — Phương Pháp Phân Tích Cao Độ

| Phương pháp | Tốc độ | Chất lượng | Ghi chú |
|-------------|--------|------------|---------|
| `pm` | ★★★★★ Nhanh nhất | ★★★ Tạm | Không cần file phụ |
| `harvest` | ★★ Chậm | ★★★★ Tốt | Ổn định giọng thấp |
| `crepe` | ★★ Chậm | ★★★★ Tốt | Cần GPU |
| `rmvpe` | ★★★★ Nhanh | ★★★★★ Tốt nhất | **Khuyến nghị**, cần `rmvpe.pt` |

---

### INDEX_RATE — Tỉ Lệ Blend FAISS
- `0.0` = không dùng index | `0.75` = mặc định | `1.0` = hoàn toàn dùng index
- Công thức: `features = index_rate × features_index + (1 - index_rate) × features_hubert`
- Chỉ có tác dụng khi `INDEX_PATH` trỏ đến file tồn tại.

---

### FILTER_RADIUS — Lọc F0
- `0-7`; chỉ tác dụng với `F0_METHOD="harvest"`
- `3` = mặc định, `0` = tắt, `7` = làm mượt mạnh.

---

### RMS_MIX_RATE — Blend Envelope Âm Lượng
- `0.0` = envelope model | `1.0` = copy envelope nguồn | `0.25` = mặc định

---

### PROTECT — Bảo Vệ Phụ Âm
- `0.33` = mặc định | `0.0` = bảo vệ tối đa | `0.5` = tắt
- Giảm nếu phụ âm ("s", "t", "p") bị artifact.

---
# BƯỚC 6: Chạy Voice Conversion

Input: `main_vocal_dereverb_path` (từ Bước 3)  
Output: `converted_vocal_path` — giọng đã chuyển sang model

In [ ]:
import soundfile as sf
import pathlib
import time

file_index = INDEX_PATH.strip() if INDEX_PATH else ""

print("Bắt đầu voice conversion...")
print(f"  Input  : {main_vocal_dereverb_path}")
print(f"  Model  : {MODEL_FILE}")
print(f"  Index  : {file_index if file_index else '(không dùng)'}")
print(f"  F0_UP_KEY={F0_UP_KEY}  F0_METHOD={F0_METHOD}  INDEX_RATE={INDEX_RATE}")
print()

t0 = time.time()

info, audio_out = vc.vc_single(
    SPEAKER_ID,
    main_vocal_dereverb_path,
    F0_UP_KEY,
    None,           # F0_FILE — không dùng
    F0_METHOD,
    file_index,
    "",             # file_index2 — không dùng
    INDEX_RATE,
    FILTER_RADIUS,
    RESAMPLE_SR,
    RMS_MIX_RATE,
    PROTECT,
)

elapsed = time.time() - t0
print("Log:", info)

if audio_out is None:
    print("\n❌ Inference thất bại. Xem log ở trên.")
    converted_vocal_path = None
else:
    tgt_sr, audio_i16 = audio_out
    converted_vocal_path = str(output_dir / f"{song_stem}_converted_vocal.wav")
    sf.write(converted_vocal_path, audio_i16, tgt_sr)

    dur = len(audio_i16) / tgt_sr
    print(f"\n✅ Voice conversion xong ({elapsed:.1f}s)")
    print(f"   Output     : {converted_vocal_path}")
    print(f"   Sample rate: {tgt_sr} Hz")
    print(f"   Thời lượng : {int(dur//60)}:{dur%60:05.2f}")
    print(f"   Tốc độ     : {dur/elapsed:.1f}x realtime")

---
# BƯỚC 7: Nghe Thử Vocal Converted

So sánh giọng gốc và giọng sau khi convert để kiểm tra chất lượng trước khi mixing.

In [ ]:
from IPython.display import Audio, display

print("🎤 Main Vocals gốc (DeReverb):")
display(Audio(main_vocal_dereverb_path))

print(f"\n🤖 Vocal sau convert (model: {MODEL_FILE}):")
if converted_vocal_path:
    display(Audio(converted_vocal_path))
else:
    print("❌ Chưa có file — chạy Bước 6 trước.")

---
# BƯỚC 8: Giải Thích Tham Số Audio Mixing

Tất cả tham số mixing đã được đặt ở **Bước 1**. Phần này giải thích để bạn điều chỉnh nếu cần.

---

## Effects Chain (áp dụng lên Converted Vocal)

```
[Converted Vocal] → HighpassFilter → Compressor → Reverb → [AI Vocal Wet]
```

### HighpassFilter
- Loại bỏ tần số thấp (rumble, proximity effect) trong giọng
- Không có tham số — luôn bật

### Compressor
- **ratio=4**: Khi âm thanh vượt threshold, giảm 4dB cho mỗi 1dB vượt
- **threshold=-15dB**: Ngưỡng kích hoạt compressor
- Cố định, không điều chỉnh được qua notebook

### Reverb

| Tham số | Giá trị mặc định | Ý nghĩa |
|---------|------------------|---------|
| `REVERB_ROOM_SIZE` | `0.15` | 0.0-1.0: kích thước không gian ảo (nhỏ=phòng nhỏ, to=sân khấu lớn) |
| `REVERB_WET` | `0.2` | 0.0-1.0: mức reverb trộn vào (càng cao càng vang) |
| `REVERB_DRY` | `0.8` | 0.0-1.0: mức signal khô giữ lại |
| `REVERB_DAMPING` | `0.7` | 0.0-1.0: tắt dần tần số cao (cao=reverb tối hơn) |

> **Mẹo:** `wet + dry` không cần bằng 1.0. `wet=0.2, dry=0.8` là cân bằng tự nhiên.

---

## Gain Mặc Định và Điều Chỉnh

| Track | Base gain | Tham số chỉnh thêm | Công thức |
|-------|-----------|--------------------|-----------|
| Main Vocal (converted+effects) | **-4 dB** | `MAIN_GAIN` | `-4 + MAIN_GAIN` dB |
| Backup Vocals | **-6 dB** | `BACKUP_GAIN` | `-6 + BACKUP_GAIN` dB |
| Instrumental | **-7 dB** | `INST_GAIN` | `-7 + INST_GAIN` dB |

Ví dụ: `MAIN_GAIN=2` → main vocal ở `-4 + 2 = -2 dB` (to hơn mặc định).  
Ví dụ: `INST_GAIN=-3` → instrumental ở `-7 - 3 = -10 dB` (nhỏ hơn mặc định).

---
# BƯỚC 9: Áp Dụng Audio Effects Lên Vocal Converted

In [ ]:
from pedalboard import Compressor, HighpassFilter, Pedalboard, Reverb
from pedalboard.io import AudioFile
import time

if converted_vocal_path is None:
    raise RuntimeError("Chưa có converted_vocal_path — chạy Bước 6 trước.")

ai_vocal_wet_path = str(output_dir / f"{song_stem}_ai_vocal_wet.wav")

board = Pedalboard([
    HighpassFilter(),
    Compressor(ratio=4, threshold_db=-15),
    Reverb(
        room_size=REVERB_ROOM_SIZE,
        dry_level=REVERB_DRY,
        wet_level=REVERB_WET,
        damping=REVERB_DAMPING,
    ),
])

print("Áp dụng effects chain: HighpassFilter → Compressor → Reverb")
print(f"  Input  : {converted_vocal_path}")
print(f"  Output : {ai_vocal_wet_path}")
print(f"  Reverb : room={REVERB_ROOM_SIZE}  wet={REVERB_WET}  dry={REVERB_DRY}  damp={REVERB_DAMPING}")

t0 = time.time()
with AudioFile(converted_vocal_path) as f:
    with AudioFile(ai_vocal_wet_path, "w", f.samplerate, f.num_channels) as o:
        while f.tell() < f.frames:
            chunk = f.read(int(f.samplerate))
            effected = board(chunk, f.samplerate, reset=False)
            o.write(effected)

elapsed = time.time() - t0
print(f"\n✅ Effects xong ({elapsed:.1f}s)  →  {ai_vocal_wet_path}")

---
# BƯỚC 10: Mix Tất Cả Tracks Thành Bài Cover Hoàn Chỉnh

Overlay 3 tracks theo thứ tự:
```
AI Vocal (wet, -4dB + MAIN_GAIN)
+ Backup Vocals  (-6dB + BACKUP_GAIN)
+ Instrumental   (-7dB + INST_GAIN)
= Final Cover
```

In [ ]:
from pydub import AudioSegment
import time

final_mix_path = str(output_dir / f"{song_stem}_Final_Cover.{OUTPUT_FORMAT}")

print("Mixing tracks...")
print(f"  Main Vocal  (wet) : {os.path.basename(ai_vocal_wet_path)}  → base -4 dB + {MAIN_GAIN:+} dB")
print(f"  Backup Vocals     : {os.path.basename(backup_vocal_path)}  → base -6 dB + {BACKUP_GAIN:+} dB")
print(f"  Instrumental      : {os.path.basename(instrumental_path)}  → base -7 dB + {INST_GAIN:+} dB")
print(f"  Output format     : {OUTPUT_FORMAT}")

t0 = time.time()

main_audio   = AudioSegment.from_file(ai_vocal_wet_path)  - 4 + MAIN_GAIN
backup_audio = AudioSegment.from_file(backup_vocal_path)  - 6 + BACKUP_GAIN
inst_audio   = AudioSegment.from_file(instrumental_path)  - 7 + INST_GAIN

# Overlay theo thứ tự: main vocal là nền, thêm backup và instrumental lên
mixed = main_audio.overlay(backup_audio).overlay(inst_audio)
mixed.export(final_mix_path, format=OUTPUT_FORMAT)

elapsed = time.time() - t0

import os
size_mb = os.path.getsize(final_mix_path) / 1024**2
print(f"\n✅ Mix xong ({elapsed:.1f}s)")
print(f"   Output : {final_mix_path}")
print(f"   Size   : {size_mb:.1f} MB")
print(f"   Thời lượng: {len(mixed)/1000:.1f} giây  |  SR: {mixed.frame_rate} Hz")

---
# BƯỚC 11: Kết Quả Cuối Cùng và So Sánh

In [ ]:
from IPython.display import Audio, display
import pathlib

print("=" * 55)
print("SO SÁNH ĐẦU VÀO / ĐẦU RA")
print("=" * 55)

print("\n🎵 Bài hát GỐC:")
display(Audio(INPUT_SONG))

print(f"\n🤖 AI Vocal Wet (vocal đã convert + effects):")
display(Audio(ai_vocal_wet_path))

print(f"\n🏆 FINAL COVER ({OUTPUT_FORMAT.upper()}):")
if pathlib.Path(final_mix_path).exists():
    display(Audio(final_mix_path))
    print(f"   → {final_mix_path}")
else:
    print("❌ Chưa có file — chạy Bước 10 trước.")

print()
print("=" * 55)
print("CÁC FILE ĐÃ TẠO")
print("=" * 55)
files_to_show = [
    ("Separation/Instrumental",   instrumental_path),
    ("Separation/Backup Vocals",  backup_vocal_path),
    ("Separation/Main DeReverb",  main_vocal_dereverb_path),
    ("RVC/Converted Vocal",       converted_vocal_path),
    ("Mixing/AI Vocal Wet",       ai_vocal_wet_path),
    ("Mixing/Final Cover",        final_mix_path),
]
for label, fpath in files_to_show:
    if fpath and pathlib.Path(fpath).exists():
        mb = pathlib.Path(fpath).stat().st_size / 1024**2
        print(f"  ✓  {label:<30s} {pathlib.Path(fpath).name}  ({mb:.1f} MB)")
    else:
        print(f"  -  {label:<30s} (chưa tạo)")

---
# BƯỚC 12: Quick Run — Toàn Bộ Pipeline Trong 1 Lần

Sau khi đã chạy qua đầy đủ từ Bước 0-11 **ít nhất 1 lần** (để setup env, load model),  
lần sau chỉ cần:
1. Sửa `INPUT_SONG` và/hoặc các tham số ở Bước 1.
2. Chạy ô dưới đây.

> **Yêu cầu:** Model đã được load (Bước 4) — nếu restart kernel, phải chạy lại Bước 2 và 4 trước.

In [ ]:
import os
import pathlib
import time
import soundfile as sf
from pedalboard import Compressor, HighpassFilter, Pedalboard, Reverb
from pedalboard.io import AudioFile
from pydub import AudioSegment
from IPython.display import Audio, display

# ============================================================
# Sửa INPUT_SONG và tham số ở Bước 1 trước khi chạy Quick Run
# ============================================================

print("=" * 55)
print("QUICK RUN — FULL PIPELINE")
print("=" * 55)
print(f"  Input  : {INPUT_SONG}")
print(f"  Model  : {MODEL_FILE}")
print(f"  Output : {OUTPUT_DIR}/")
print()

qr_start = time.time()

# Setup
input_song_path = pathlib.Path(INPUT_SONG)
if not input_song_path.exists():
    raise FileNotFoundError(f"File không tồn tại: {INPUT_SONG}")

qr_stem = input_song_path.stem
qr_out = STANDALONE_ROOT / OUTPUT_DIR
qr_out.mkdir(parents=True, exist_ok=True)
qr_sep = str(qr_out / "separation")
qr_mdx_dir = pathlib.Path(MDX_MODELS_DIR).resolve()

# --- Stage 1 ---
print("[1/5] Stage 1: Tách Instrumental + Vocals...")
t = time.time()
qr_vocals, qr_inst = run_mdx(
    qr_sep, qr_mdx_dir / "UVR-MDX-NET-Voc_FT.onnx", INPUT_SONG, denoise=DENOISE)
print(f"      Xong ({time.time()-t:.1f}s)")

# --- Stage 2 ---
print("[2/5] Stage 2: Tách Backup + Main Vocals...")
t = time.time()
qr_backup, qr_main = run_mdx(
    qr_sep, qr_mdx_dir / "UVR_MDXNET_KARA_2.onnx", qr_vocals,
    suffix="Backup", invert_suffix="Main", denoise=DENOISE)
os.remove(qr_vocals)
print(f"      Xong ({time.time()-t:.1f}s)")

# --- Stage 3 ---
print("[3/5] Stage 3: De-reverb Main Vocals...")
t = time.time()
_, qr_dereverb = run_mdx(
    qr_sep, qr_mdx_dir / "Reverb_HQ_By_FoxJoy.onnx", qr_main,
    invert_suffix="DeReverb", exclude_main=True, denoise=DENOISE)
os.remove(qr_main)
print(f"      Xong ({time.time()-t:.1f}s)")

# --- Voice Conversion ---
print("[4/5] Voice Conversion (RVC)...")
t = time.time()
file_index = INDEX_PATH.strip() if INDEX_PATH else ""
info, audio_out = vc.vc_single(
    SPEAKER_ID, qr_dereverb, F0_UP_KEY, None, F0_METHOD,
    file_index, "", INDEX_RATE, FILTER_RADIUS, RESAMPLE_SR, RMS_MIX_RATE, PROTECT,
)
if audio_out is None:
    raise RuntimeError(f"Voice conversion thất bại: {info}")
tgt_sr, audio_i16 = audio_out
qr_converted = str(qr_out / f"{qr_stem}_converted_vocal.wav")
sf.write(qr_converted, audio_i16, tgt_sr)
print(f"      Xong ({time.time()-t:.1f}s)")

# --- Effects + Mix ---
print("[5/5] Audio Effects + Mix...")
t = time.time()

qr_wet = str(qr_out / f"{qr_stem}_ai_vocal_wet.wav")
board = Pedalboard([
    HighpassFilter(),
    Compressor(ratio=4, threshold_db=-15),
    Reverb(room_size=REVERB_ROOM_SIZE, dry_level=REVERB_DRY,
           wet_level=REVERB_WET, damping=REVERB_DAMPING),
])
with AudioFile(qr_converted) as f:
    with AudioFile(qr_wet, "w", f.samplerate, f.num_channels) as o:
        while f.tell() < f.frames:
            chunk = f.read(int(f.samplerate))
            o.write(board(chunk, f.samplerate, reset=False))

qr_final = str(qr_out / f"{qr_stem}_Final_Cover.{OUTPUT_FORMAT}")
main_a   = AudioSegment.from_file(qr_wet)   - 4 + MAIN_GAIN
backup_a = AudioSegment.from_file(qr_backup) - 6 + BACKUP_GAIN
inst_a   = AudioSegment.from_file(qr_inst)   - 7 + INST_GAIN
mixed    = main_a.overlay(backup_a).overlay(inst_a)
mixed.export(qr_final, format=OUTPUT_FORMAT)
print(f"      Xong ({time.time()-t:.1f}s)")

total = time.time() - qr_start
print()
print("=" * 55)
print(f"✅ PIPELINE HOÀN TẤT trong {total:.1f}s ({total/60:.1f} phút)")
print("=" * 55)
print(f"   FINAL COVER : {qr_final}")
print()

print("🏆 KẾT QUẢ:")
display(Audio(qr_final))

---
# Xử Lý Sự Cố Thường Gặp

| Vấn đề | Nguyên nhân | Cách khắc phục |
|--------|-------------|----------------|
| Giọng sai tông | F0_UP_KEY chưa đúng | Điều chỉnh `F0_UP_KEY` ±1 |
| Giọng nghe "robotic" | Train ít epoch | Train thêm hoặc tăng data |
| Phụ âm bị vỡ | PROTECT quá cao | Giảm `PROTECT` về 0.1-0.2 |
| Giọng không giống model | INDEX_RATE thấp | Tăng `INDEX_RATE` lên 0.75-0.9 |
| Artifact nhiều | INDEX_RATE quá cao | Giảm `INDEX_RATE` xuống 0.3-0.5 |
| Nhạc nền quá to/nhỏ | INST_GAIN chưa phù hợp | Tăng/giảm `INST_GAIN` |
| Reverb quá nhiều/ít | REVERB_WET chưa phù hợp | Giảm/tăng `REVERB_WET` (mặc định 0.2) |
| OOM VRAM | File quá dài hoặc VRAM thiếu | Chia audio thành đoạn ngắn |
| MDX hash không khớp | Sai file ONNX | Kiểm tra lại các file trong `MDX_MODELS_DIR` |
| `RMVPE` không load | Thiếu `assets/rmvpe/rmvpe.pt` | Dùng `F0_METHOD="pm"` tạm thời |

## Gợi ý tham số theo use case

| Use case | F0_UP_KEY | F0_METHOD | INDEX_RATE | PROTECT | MAIN_GAIN |
|----------|-----------|-----------|------------|---------|----------|
| Hát (cùng giới tính) | 0 | rmvpe | 0.75 | 0.33 | 0 |
| Hát nam → nữ | +5~+7 | rmvpe | 0.75 | 0.33 | 0 |
| Hát nữ → nam | -5~-7 | rmvpe | 0.75 | 0.33 | 0 |
| Chất lượng tối đa | 0 | rmvpe | 0.85 | 0.25 | +1 |
| Vocal quá nhỏ trong mix | 0 | rmvpe | 0.75 | 0.33 | +3~+5 |